In [ ]:
import logging

from tqdm import tqdm
import pyarrow as pa

from basic_rag.data import DatasetFactory
from basic_rag.chunking import ChunkerFactory
from basic_rag.embeddings import EmbedderFactory
from basic_rag.index import VectorDBFactory
from basic_rag.retrieval import RetrieverFactory
from basic_rag.evaluation import Evaluator
from basic_rag.reranking import ReRankerFactory

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
"""Configs"""

# Dataset configs
dataset_type = "beir"
dataset_name = "scifact"
path_qrel = f"../datasets/beir/datasets/{dataset_name}/qrels/train.tsv"
path_corpus = f"../datasets/beir/datasets/{dataset_name}/corpus.jsonl"
path_queries = f"../datasets/beir/datasets/{dataset_name}/queries.jsonl"
batch_size = 512

# Chunker configs
chunker_type = "token"
token_chunker_encoder = "cl100k_base"
chunk_size = 1024
chunk_overlap = 50

# Embedder config
embedder_type = "openai"
embedding_model = "text-embedding-3-small"

# Indexer config
indexer_type = "lancedb"
index_type = "IVF_PQ"
vector_db_uri = "lancedb_index"
vector_db_name = (
    f"{dataset_type}_{dataset_name}_{chunker_type}_{embedding_model}_{index_type}"
)
load_existing = True

# Reranker config
re_ranker_type = "cross_encoder"
re_rank_model = "cross-encoder/ms-marco-MiniLM-L6-v2"

# Retriever config
retriever_type = indexer_type
dist_metric = "l2"
retrieval_limit = 10

In [ ]:
# Process documents
dataset = DatasetFactory.create(dataset_type)
vector_db = VectorDBFactory.create(
    indexer_type,
    db_name=vector_db_name,
    db_uri=vector_db_uri,
    load_existing=load_existing,
)
embedder = EmbedderFactory.create(embedder_type, model=embedding_model)

if not load_existing:
    chunker = ChunkerFactory.create(
        chunker_type,
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        encoding=token_chunker_encoder,
    )
    documents_loader = dataset.document_loader(file_path=path_corpus, batch_size=batch_size)

    # Stream documents in batches
    for idx, batch in tqdm(enumerate(documents_loader)):

        # Chunk
        chunks = chunker.chunk(documents=batch)
        logger.debug(f"chunking docs with ID: {[c.doc_id for c in chunks]}")

        # Embedding
        embeddings = embedder.embed_chunks(chunks)
        logger.debug(
            f"embedding chunks with doc_id: {[e.chunk.doc_id for e in embeddings]}"
        )

        # Add to index
        vector_db.add_to_db(embeddings)
        logger.debug(f"added batch {idx} to index")

    vector_db.index_db(index_type=index_type)

In [ ]:
vector_db.get_db().to_pandas().head()

In [ ]:
# Retrieval
query_loader = dataset.query_loader(path_queries, batch_size=batch_size)
retriever = RetrieverFactory.create(retriever_type, vector_db_uri, vector_db_name)
reranker = ReRankerFactory.create(re_ranker_type, re_rank_model)

# Stream queries in batches.
retrievals = []
for _, batch in tqdm(enumerate(query_loader)):
    embeddings = embedder.embed_queries(batch)
    batch_results = retriever.search_queries(
        q_embeddings=embeddings,
        metric=dist_metric,
        limit=retrieval_limit,
    )

    batch_results = reranker.re_rank_batch(embeddings, batch_results)

    retrievals.append(batch_results)

    break

retrievals = pa.concat_tables(retrievals)
retrievals_df = retrievals.to_pandas()

In [ ]:
# Evaluation
evaluator = Evaluator()

# Convert to pytrec_eval format
run = evaluator.df_to_pytrec(retrievals_df)

with open(path_qrel, "r") as f_qrel:
    qrel = evaluator.parse_qrel(f_qrel, ignore_header=True)

# evaluate
measures = [
    "map_cut.1",
    "ndcg_cut.1",
    "recall.1",
    "P.1",
    "map_cut.5",
    "ndcg_cut.5",
    "recall.5",
    "P.5",
    "map_cut.10",
    "ndcg_cut.10",
    "recall.10",
    "P.10",
]

results = evaluator.evaluate(qrel, run, measures)
logger.info("Evalutation completed!")